# Mage Iris pipeline experiments
Chaitanya Kumar · 23BCS064 · CS-8037

This notebook executes the real Mage CLI. The baseline uses the lecture split (`test_size=0.2`, `random_state=42`), decision tree (`random_state=42`) and an accuracy threshold of 0.8. CSV and model files are removed before each depth experiment so a stale file cannot masquerade as a successful export. The evaluation branch and tested metrics exporter are assignment extensions.

The first code cell creates the project only where files are absent. In Google Colab it installs dependencies in a separate Python 3.12 environment.

In [1]:
import os,sys,subprocess,json
from pathlib import Path
COLAB = 'COLAB_RELEASE_TAG' in os.environ
ROOT = Path('/content/freshcart-pipelines') if COLAB else Path.cwd()
if not COLAB and ROOT.name == 'iris_project': ROOT=ROOT.parent
ROOT.mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
FILES = {'iris_project/data_loaders/__init__.py': '', 'iris_project/data_loaders/load_iris.py': "import pandas as pd\nfrom sklearn.datasets import load_iris\n\n# One-line version for students: “If Mage hasn’t already provided the data_loader\n# and test labels, import them, so the file works both inside and outside Mage.\n\n# Import Mage's decorators only if they aren't already loaded\nif 'data_loader' not in globals():\n    from mage_ai.data_preparation.decorators import data_loader\nif 'test' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n\n# data_loader marks this function as the pipeline's first step: it brings data in\n# It loads the built-in Iris dataset from scikit-learn as a pandas table\n# The returned DataFrame is passed automatically to the next block (prepare_data)\n@data_loader\ndef load_data(*args, **kwargs):\n    iris = load_iris(as_frame=True)\n    df = iris.frame                      # 4 feature columns + 'target'\n    print('Rows loaded:', len(df))\n    return df\n\n\n# test marks this as a quality check that Mage runs right after the block finishes\n# 'output' is whatever load_data returned, so we can verify the data\n# If the table doesn't have exactly 150 rows, the block fails and the pipeline stops\n@test\ndef test_output(output, *args):\n    assert len(output) == 150, 'Iris must have 150 rows'", 'iris_project/transformers/evaluate_model.py': "import pandas as pd\nfrom sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score\n\nif 'transformer' not in globals():\n    from mage_ai.data_preparation.decorators import transformer\n\n\n@transformer\ndef evaluate(df, *args, **kwargs):\n    y_true = df['target']\n    y_pred = df['predicted']\n\n    print('Confusion matrix:')\n    print(pd.crosstab(y_true, y_pred, rownames=['actual'], colnames=['predicted']))\n\n    metrics = pd.DataFrame({\n        'metric': ['accuracy', 'precision', 'recall', 'f1_score'],\n        'value': [\n            round(accuracy_score(y_true, y_pred), 3),\n            round(precision_score(y_true, y_pred, average='macro'), 3),\n            round(recall_score(y_true, y_pred, average='macro'), 3),\n            round(f1_score(y_true, y_pred, average='macro'), 3),\n        ]\n    })\n    print(metrics.to_string(index=False))\n    return metrics\nif 'test' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n@test\ndef test_metrics(output, *args):\n    assert output['metric'].tolist() == ['accuracy', 'precision', 'recall', 'f1_score']\n    assert output['value'].between(0, 1).all(), 'Metrics must be in [0, 1]'\n", 'iris_project/transformers/train_model.py': "import joblib\nfrom sklearn.tree import DecisionTreeClassifier\n\n# Import Mage's decorators only if they aren't already loaded\nif 'transformer' not in globals():\n    from mage_ai.data_preparation.decorators import transformer\nif 'test' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n# The 4 measurement columns the model uses to predict the species\nFEATURES = ['sepal length (cm)', 'sepal width (cm)',\n            'petal length (cm)', 'petal width (cm)']\n\n\n# transformer receives prepare_data's table (150 rows with 'split') as 'df'\n# Reads max_depth from Mage's pipeline variables, then trains a decision tree on the train rows\n# Predicts the 30 test rows, prints accuracy, saves the model and returns the predictions\n@transformer\ndef train(df, *args, **kwargs):\n    max_depth = int(kwargs.get('max_depth', 3))\n\n    train_df = df[df['split'] == 'train']\n    test_df = df[df['split'] == 'test'].copy()\n\n    model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)\n    model.fit(train_df[FEATURES], train_df['target'])\n\n    test_df['predicted'] = model.predict(test_df[FEATURES])\n    accuracy = (test_df['predicted'] == test_df['target']).mean()\n    print('max_depth =', max_depth, '| Test accuracy =', round(accuracy, 3))\n\n    joblib.dump(model, 'iris_model.pkl')\n    return test_df\n\n\n# Runs right after train finishes, using its 30 predicted rows as 'output'\n# Recalculates accuracy by comparing 'predicted' with the true 'target'\n# If accuracy is below 80%, the block fails and save_results / evaluate_model never run\n@test\ndef test_accuracy(output, *args):\n    accuracy = (output['predicted'] == output['target']).mean()\n    assert accuracy >= 0.8, 'Model accuracy is below 80%'", 'iris_project/transformers/prepare_data.py': "import pandas as pd\nfrom sklearn.model_selection import train_test_split\n\n# Import Mage's decorators only if they aren't already loaded\nif 'transformer' not in globals():\n    from mage_ai.data_preparation.decorators import transformer\nif 'test' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n\n# ransformer marks this as a step that changes data coming from an earlier block\n# 'df' is the 150-row table returned by load_iris, passed in automatically by Mage\n# It splits the data 80/20 and adds a 'split' column labelling each row train or test\n@transformer\ndef transform(df, *args, **kwargs):\n    train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)\n\n    train_df = train_df.copy()\n    test_df = test_df.copy()\n    train_df['split'] = 'train'\n    test_df['split'] = 'test'\n\n    result = pd.concat([train_df, test_df])\n    print('Train rows:', len(train_df), '| Test rows:', len(test_df))\n    return result\n\n\n# Runs right after transform finishes, using its returned table as 'output'\n# Checks that the new 'split' column was actually created\n# If it's missing, the block fails and train_model never runs\n@test\ndef test_output(output, *args):\n    assert 'split' in output.columns, 'split column missing'", 'iris_project/transformers/__init__.py': '', 'iris_project/data_exporters/__init__.py': '', 'iris_project/data_exporters/save_results.py': "# Import Mage's decorator only if it isn't already loaded\nif 'data_exporter' not in globals():\n    from mage_ai.data_preparation.decorators import data_exporter\n\n\n# data_exporter marks this as a step that sends data out of the pipeline\n# 'df' is the 30 predicted rows returned by train_model, passed in automatically by Mage\n# It saves them to a CSV file and returns nothing, because this is an end of the pipeline\n@data_exporter\ndef export_data(df, *args, **kwargs):\n    df.to_csv('iris_predictions.csv', index=False)\n    print('Saved', len(df), 'predictions to iris_predictions.csv')\nif 'test' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n@test\ndef test_predictions_written(*args):\n    from pathlib import Path\n    import pandas as pd\n    path = Path('iris_predictions.csv')\n    assert path.exists(), 'Predictions were not written'\n    saved = pd.read_csv(path)\n    assert len(saved) == 30 and {'target', 'predicted'}.issubset(saved.columns)\n", 'iris_project/data_exporters/save_metrics.py': '"""C4 extension: publish the evaluated metrics only after their quality check."""\nif \'data_exporter\' not in globals():\n    from mage_ai.data_preparation.decorators import data_exporter\nif \'test\' not in globals():\n    from mage_ai.data_preparation.decorators import test\n\n@data_exporter\ndef export_metrics(df, *args, **kwargs):\n    df.to_csv(\'iris_metrics.csv\', index=False)\n    print(\'Saved\', len(df), \'metrics to iris_metrics.csv\')\n\n@test\ndef test_metrics_written(*args):\n    import pandas as pd\n    from pathlib import Path\n    assert Path(\'iris_metrics.csv\').is_file()\n    saved = pd.read_csv(\'iris_metrics.csv\')\n    assert saved[\'metric\'].tolist() == [\'accuracy\', \'precision\', \'recall\', \'f1_score\']\n    assert saved[\'value\'].between(0, 1).all()\n', 'iris_project/pipelines/iris_classifier/metadata.yaml': 'blocks:\n- name: load_iris\n  uuid: load_iris\n  type: data_loader\n  language: python\n  upstream_blocks: []\n  downstream_blocks: [prepare_data]\n- name: prepare_data\n  uuid: prepare_data\n  type: transformer\n  language: python\n  upstream_blocks: [load_iris]\n  downstream_blocks: [train_model]\n- name: train_model\n  uuid: train_model\n  type: transformer\n  language: python\n  upstream_blocks: [prepare_data]\n  downstream_blocks: [save_results, evaluate_model]\n- name: save_results\n  uuid: save_results\n  type: data_exporter\n  language: python\n  upstream_blocks: [train_model]\n  downstream_blocks: []\n- name: evaluate_model\n  uuid: evaluate_model\n  type: transformer\n  language: python\n  upstream_blocks: [train_model]\n  downstream_blocks: [save_metrics]\n- name: save_metrics\n  uuid: save_metrics\n  type: data_exporter\n  language: python\n  upstream_blocks: [evaluate_model]\n  downstream_blocks: []\nname: iris_classifier\nuuid: iris_classifier\ntype: python\nvariables:\n  max_depth: 3\n', 'iris_project/metadata.yaml': 'project_type: standalone\nvariables_dir: ~/.mage_data\nremote_variables_dir:\nproject_uuid: 2b9a6282d9ff44ccbd5b4ec25f46941a\n', 'iris_project/run_experiments.py': '"""Run C1/C2/C3/C4 with the real Mage CLI and capture unedited output."""\nimport json, subprocess, sys, time\nfrom pathlib import Path\n\nROOT = Path(__file__).resolve().parents[1]\nEVIDENCE = ROOT / \'evidence\'\n\ndef run():\n    EVIDENCE.mkdir(exist_ok=True)\n    results = []\n    for depth in range(1, 6):\n        for filename in [\'iris_predictions.csv\', \'iris_metrics.csv\', \'iris_model.pkl\']:\n            (ROOT / filename).unlink(missing_ok=True)\n        command = [str(Path(sys.executable).with_name(\'mage\')), \'run\', \'iris_project\',\n                   \'iris_classifier\', \'--test\', \'--runtime-vars\', json.dumps({\'max_depth\': depth})]\n        started = time.time()\n        result = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE,\n                                stderr=subprocess.STDOUT, timeout=180)\n        log = result.stdout\n        (EVIDENCE / f\'C2-depth-{depth}.txt\').write_text(\'$ \' + \' \'.join(command[1:]) + \'\\n\' + log)\n        import re\n        found = re.search(r\'Test accuracy = ([0-9.]+)\', log)\n        row = {\'max_depth\': depth, \'test_accuracy\': float(found.group(1)) if found else None,\n               \'exit_code\': result.returncode, \'accuracy_gate_failed\': \'Model accuracy is below 80%\' in log,\n               \'predictions_created\': (ROOT / \'iris_predictions.csv\').exists(),\n               \'metrics_created\': (ROOT / \'iris_metrics.csv\').exists(),\n               \'elapsed_seconds\': round(time.time() - started, 2)}\n        results.append(row)\n        print(row, flush=True)\n    (EVIDENCE / \'C2-results.json\').write_text(json.dumps(results, indent=2))\n    # Leave the default successful run and its outputs visible for the notebook and UI.\n    command = [str(Path(sys.executable).with_name(\'mage\')), \'run\', \'iris_project\',\n               \'iris_classifier\', \'--test\']\n    result = subprocess.run(command, cwd=ROOT, text=True, stdout=subprocess.PIPE,\n                            stderr=subprocess.STDOUT, timeout=180)\n    (EVIDENCE / \'C1-default-run.txt\').write_text(\'$ mage run iris_project iris_classifier --test\\n\'+result.stdout)\n    print(\'Default run exit:\', result.returncode, flush=True)\n    if result.returncode: raise SystemExit(result.returncode)\n\nif __name__ == \'__main__\': run()\n', 'requirements.txt': 'ipykernel==6.15.0\nmage-ai==0.9.80\nnbclient==0.11.0\nnbformat==5.11.1\npandas==2.3.3\npika==1.4.4\npytest==9.2.0\nscikit-learn==1.7.2\nsnowflake-connector-python==4.0.0\n'}
for name,text in FILES.items():
    path=ROOT/name;path.parent.mkdir(parents=True,exist_ok=True)
    if not path.exists(): path.write_text(text)
if COLAB:
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
    subprocess.run([sys.executable,'-m','uv','venv','-p','3.12','/content/mage_env'],check=True)
    PYTHON=Path('/content/mage_env/bin/python')
    subprocess.run([sys.executable,'-m','uv','pip','install','--python',str(PYTHON),'-r','requirements.txt'],check=True)
else:
    PYTHON=Path(sys.executable)
MAGE=PYTHON.with_name('mage')
print('Execution environment:', 'Google Colab' if COLAB else 'Local Python 3.12')
print('Project files ready:',len(FILES))


Execution environment: Google Colab
Project files ready: 13


## C1 Baseline pipeline run
The six blocks are load_iris (data_loader, 150-row test), prepare_data (transformer, split-column test), train_model (transformer, accuracy test), save_results (data_exporter, CSV test), evaluate_model (transformer, metric-range test), and save_metrics (data_exporter, saved-metrics test).

In [2]:
result=subprocess.run([str(MAGE),'run','iris_project','iris_classifier','--test'],capture_output=True,text=True,cwd=ROOT)
print(result.stdout);print(result.stderr)
assert result.returncode==0


DBT library not installed.
Rows loaded: 150
--------------------------------------------------------------
1/1 tests passed.
Train rows: 120 | Test rows: 30
--------------------------------------------------------------
1/1 tests passed.
max_depth = 3 | Test accuracy = 1.0
--------------------------------------------------------------
1/1 tests passed.
Confusion matrix:
Saved 30 predictions to iris_predictions.csv
predicted   0  1   2
actual              
0          10  0   0
1           0  9   0
2           0  0  11
   metric  value
 accuracy    1.0
precision    1.0
   recall    1.0
 f1_score    1.0
--------------------------------------------------------------
1/1 tests passed.
--------------------------------------------------------------
1/1 tests passed.
Saved 4 metrics to iris_metrics.csv
--------------------------------------------------------------
1/1 tests passed.
Pipeline run completed.

INFO:mage_ai.data_preparation.executors.pipeline_executor:[PipelineExecutor] No running 

## C2 Accuracy gate experiments
Run all depths, with `--runtime-vars` and a fresh predictions path before each run. A failing training test must prevent both downstream branches from exporting results.

In [3]:
result=subprocess.run([str(PYTHON),'iris_project/run_experiments.py'],capture_output=True,text=True,cwd=ROOT)
print(result.stdout);print(result.stderr)
assert result.returncode==0
import pandas as pd
rows=json.loads((ROOT/'evidence/C2-results.json').read_text())
display(pd.DataFrame(rows))
assert rows[0]['accuracy_gate_failed'] and not rows[0]['predictions_created']
assert all(r['predictions_created'] for r in rows[1:])


{'max_depth': 1, 'test_accuracy': 0.633, 'exit_code': 1, 'accuracy_gate_failed': True, 'predictions_created': False, 'metrics_created': False, 'elapsed_seconds': 10.21}
{'max_depth': 2, 'test_accuracy': 0.967, 'exit_code': 0, 'accuracy_gate_failed': False, 'predictions_created': True, 'metrics_created': True, 'elapsed_seconds': 4.34}
{'max_depth': 3, 'test_accuracy': 1.0, 'exit_code': 0, 'accuracy_gate_failed': False, 'predictions_created': True, 'metrics_created': True, 'elapsed_seconds': 4.34}
{'max_depth': 4, 'test_accuracy': 1.0, 'exit_code': 0, 'accuracy_gate_failed': False, 'predictions_created': True, 'metrics_created': True, 'elapsed_seconds': 5.39}
{'max_depth': 5, 'test_accuracy': 1.0, 'exit_code': 0, 'accuracy_gate_failed': False, 'predictions_created': True, 'metrics_created': True, 'elapsed_seconds': 4.29}
Default run exit: 0




,max_depth,test_accuracy,exit_code,accuracy_gate_failed,predictions_created,metrics_created,elapsed_seconds
0,1,0.633,1,True,False,False,10.21
1,2,0.967,0,False,True,True,4.34
2,3,1.000,0,False,True,True,4.34
3,4,1.000,0,False,True,True,5.39
4,5,1.000,0,False,True,True,4.29


## C3 Evaluation branch and metrics
`train_model` has two downstream blocks: `save_results` and `evaluate_model`. The evaluation branch computes macro precision, recall and F1 alongside accuracy. The built-in Iris holdout contains only 30 rows; perfect scores on that holdout are not proof of general performance.

In [4]:
print((ROOT/'iris_project/pipelines/iris_classifier/metadata.yaml').read_text())
predictions=pd.read_csv(ROOT/'iris_predictions.csv')
display(pd.crosstab(predictions['target'],predictions['predicted'],rownames=['actual'],colnames=['predicted']))
display(pd.read_csv(ROOT/'iris_metrics.csv'))


blocks:
- name: load_iris
  uuid: load_iris
  type: data_loader
  language: python
  upstream_blocks: []
  downstream_blocks: [prepare_data]
- name: prepare_data
  uuid: prepare_data
  type: transformer
  language: python
  upstream_blocks: [load_iris]
  downstream_blocks: [train_model]
- name: train_model
  uuid: train_model
  type: transformer
  language: python
  upstream_blocks: [prepare_data]
  downstream_blocks: [save_results, evaluate_model]
- name: save_results
  uuid: save_results
  type: data_exporter
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: []
- name: evaluate_model
  uuid: evaluate_model
  type: transformer
  language: python
  upstream_blocks: [train_model]
  downstream_blocks: [save_metrics]
- name: save_metrics
  uuid: save_metrics
  type: data_exporter
  language: python
  upstream_blocks: [evaluate_model]
  downstream_blocks: []
name: iris_classifier
uuid: iris_classifier
type: python
variables:
  max_depth: 3



predicted,0,1,2
actual,,,
0,10,0,0
1,0,9,0
2,0,0,11


,metric,value
0,accuracy,1.0
1,precision,1.0
2,recall,1.0
3,f1_score,1.0


## C4 Tested metrics exporter
The new `save_metrics` data_exporter is downstream of `evaluate_model`. It writes four metrics to `iris_metrics.csv`; its own test verifies that all four metric names exist and every value lies in [0,1].

In [5]:
print((ROOT/'iris_project/data_exporters/save_metrics.py').read_text())
metrics=pd.read_csv(ROOT/'iris_metrics.csv')
assert metrics['metric'].tolist()==['accuracy','precision','recall','f1_score']
assert metrics['value'].between(0,1).all()
print('C4 saved metrics test: PASS')


"""C4 extension: publish the evaluated metrics only after their quality check."""
if 'data_exporter' not in globals():
    from mage_ai.data_preparation.decorators import data_exporter
if 'test' not in globals():
    from mage_ai.data_preparation.decorators import test

@data_exporter
def export_metrics(df, *args, **kwargs):
    df.to_csv('iris_metrics.csv', index=False)
    print('Saved', len(df), 'metrics to iris_metrics.csv')

@test
def test_metrics_written(*args):
    import pandas as pd
    from pathlib import Path
    assert Path('iris_metrics.csv').is_file()
    saved = pd.read_csv('iris_metrics.csv')
    assert saved['metric'].tolist() == ['accuracy', 'precision', 'recall', 'f1_score']
    assert saved['value'].between(0, 1).all()

C4 saved metrics test: PASS


## Open the real Mage UI in Google Colab
This reproduction cell starts Mage with a newly generated temporary password, then opens a Cloudflare Quick Tunnel. The graph was actually opened and captured in `evidence/C3-cloudflare-mage-graph.jpg`. The first email-gated tunnel was unavailable through the browser gateway; Mage authentication was used for the working preview. Output from this setup cell is cleared to keep the temporary password out of the deliverable. Run it only when another preview is needed, save the password privately for that session, then run the cleanup cell.


In [ ]:
if COLAB:
    import time, re, secrets, urllib.request
    for process_name in ['tunnel', 'server']:
        process = globals().get(process_name)
        if process is not None:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill()
    download = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64'
    urllib.request.urlretrieve(download, '/content/cloudflared')
    Path('/content/cloudflared').chmod(0o755)
    preview_password = secrets.token_urlsafe(32)
    preview_database = '/content/mage-preview-' + secrets.token_hex(8) + '.db'
    env = os.environ.copy()
    env.update(REQUIRE_USER_AUTHENTICATION='True',
               DEFAULT_OWNER_EMAIL='23bcs064@iiitdmj.ac.in',
               DEFAULT_OWNER_PASSWORD=preview_password,
               DEFAULT_OWNER_USERNAME='chaitanya',
               MAGE_DATABASE_CONNECTION_URL='sqlite:///' + preview_database)
    server = subprocess.Popen([str(MAGE), 'start', 'iris_project', '--host', '127.0.0.1', '--port', '6789'],
                              cwd=ROOT, env=env, stdout=open('/content/secure-mage.log', 'w'), stderr=subprocess.STDOUT)
    for attempt in range(90):
        try:
            urllib.request.urlopen('http://127.0.0.1:6789', timeout=2)
            break
        except Exception: time.sleep(1)
    else: raise RuntimeError('Mage failed to start; inspect secure-mage.log privately')
    tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:6789', '--no-autoupdate'],
                              stdout=open('/content/secure-tunnel.log', 'w'), stderr=subprocess.STDOUT)
    urls = []
    for attempt in range(45):
        text = Path('/content/secure-tunnel.log').read_text()
        urls = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
        if urls: break
        time.sleep(1)
    print('Mage URL:', urls[0] if urls else 'Not ready')
    print('Preview username: 23bcs064@iiitdmj.ac.in')
    print('Temporary preview password:', preview_password)
    print('After the screenshot: stop services, then clear this cell output before saving or sharing.')
else:
    print('UI preview requires Google Colab; use mage start locally for a local preview.')


## Stop the temporary services after the graph screenshot
Run this cleanup cell after saving the screenshot.

In [9]:
for process in [tunnel, server]:
    process.terminate()
    try: process.wait(timeout=10)
    except subprocess.TimeoutExpired: process.kill()
print('Temporary Mage server and Cloudflare tunnel stopped.')

Temporary Mage server and Cloudflare tunnel stopped.
